# 15. Sevilla Mention Sentiment — Sentimiento por mención de plato

Este notebook construye la primera versión del cálculo de sentimiento por mención de plato para el prototipo serio de Hidden Gems Sevilla.

Entrada principal:

```text
data/artifacts/ai/sevilla/dish_normalization/sevilla_dish_candidates_ranking_ready_v1.jsonl
```

Salida principal:

```text
data/artifacts/ai/sevilla/sentiment/sevilla_dish_mentions_with_sentiment_v1.jsonl
```

El enfoque es híbrido y trazable:

1. usa contexto local de la mención;
2. aplica lexicones gastronómicos positivos/negativos en español;
3. detecta negaciones, intensificadores y contraste;
4. usa el rating de la review solo como fallback de baja confianza;
5. genera flags para revisar casos ambiguos;
6. conserva todas las claves necesarias para agregación y ranking posterior.

In [1]:
# ============================================================
# 01. Imports y configuración general
# ============================================================

from __future__ import annotations

import json
import math
import re
import unicodedata
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 260)
pd.set_option("display.width", 180)

NOTEBOOK_NAME = "15_sevilla_mention_sentiment"
VERSION = "sevilla_mention_sentiment_hybrid_v1"

print("Notebook:", NOTEBOOK_NAME)
print("Version:", VERSION)


Notebook: 15_sevilla_mention_sentiment
Version: sevilla_mention_sentiment_hybrid_v1


In [2]:
# ============================================================
# 02. Rutas del proyecto
# ============================================================

def find_project_root(start: Optional[Path] = None) -> Path:
    """Busca la raíz del proyecto a partir del directorio actual."""
    start = start or Path.cwd()
    candidates = [start] + list(start.parents)
    for candidate in candidates:
        if (candidate / "data").exists() and ((candidate / "scripts").exists() or (candidate / "notebooks").exists()):
            return candidate
    return start

PROJECT_DIR = find_project_root()

INPUT_PATH = PROJECT_DIR / "data" / "artifacts" / "ai" / "sevilla" / "dish_normalization" / "sevilla_dish_candidates_ranking_ready_v1.jsonl"
OUTPUT_DIR = PROJECT_DIR / "data" / "artifacts" / "ai" / "sevilla" / "sentiment"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("INPUT_PATH:", INPUT_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"No existe el archivo de entrada: {INPUT_PATH}")


PROJECT_DIR: c:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline
INPUT_PATH: c:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline\data\artifacts\ai\sevilla\dish_normalization\sevilla_dish_candidates_ranking_ready_v1.jsonl
OUTPUT_DIR: c:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline\data\artifacts\ai\sevilla\sentiment


In [3]:
# ============================================================
# 03. Funciones de lectura/escritura JSONL
# ============================================================

def read_jsonl(path: Path) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"JSON inválido en línea {line_number}: {exc}") from exc
    return rows


def write_jsonl(rows: Iterable[dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")


def save_json(data: dict[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2, default=str)


def safe_value(value: Any) -> Any:
    if isinstance(value, (list, dict, tuple)):
        return value
    try:
        if pd.isna(value):
            return None
    except TypeError:
        pass
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    return value


def dataframe_to_records(df_: pd.DataFrame) -> list[dict[str, Any]]:
    records = []
    for row in df_.to_dict(orient="records"):
        records.append({k: safe_value(v) for k, v in row.items()})
    return records


In [4]:
# ============================================================
# 04. Carga y validación básica del dataset
# ============================================================

rows = read_jsonl(INPUT_PATH)
df = pd.DataFrame(rows)

print("Menciones ranking-ready cargadas:", len(df))
print("Columnas:", len(df.columns))
display(df.head(3))

required_columns = [
    "mention_id",
    "dish_id_v1",
    "review_id",
    "place_id",
    "place_name",
    "district_name",
    "neighborhood_name",
    "rating_value",
    "review_sentiment_from_rating",
    "dish_mention_text",
    "display_dish_name_es_v1",
    "normalized_dish_name_es_v1",
    "context_sentence",
    "window_context",
    "text",
]

missing = [col for col in required_columns if col not in df.columns]
if missing:
    raise ValueError(f"Faltan columnas obligatorias: {missing}")

print("IDs únicos mention_id:", df["mention_id"].nunique() == len(df))
print("Reviews únicas:", df["review_id"].nunique())
print("Locales únicos:", df["place_id"].nunique())
print("Platos únicos:", df["dish_id_v1"].nunique())
print("Barrios únicos:", df["neighborhood_name"].nunique())
print("Distritos únicos:", df["district_name"].nunique())


Menciones ranking-ready cargadas: 2979
Columnas: 43


,mention_id,dish_id_v1,review_id,place_id,place_source_ref_id,source_review_id,source_place_record_id,place_name,district_id,district_name,neighborhood_id,neighborhood_name,rating_value,review_sentiment_from_rating,review_language,dish_mention_text,dish_mention_normalized,canonical_dish_name_es_v1,normalized_dish_name_es_v1,display_dish_name_es_v1,dish_group_es_v1,dish_family_es_v1,dish_specificity_v1,candidate_type,lexicon_group,detection_method,pattern_name,confidence,mention_quality_tier_v1,ranking_eligible_v1,eligibility_status_v1,eligibility_reason_v1,needs_manual_review_v1,normalization_method_v1,start_char,end_char,context_sentence,window_context,has_order_trigger,has_positive_context,has_negative_context,has_contrast_context,text
0,1fde28a5f80c28e3948ba1bf865c0b74017762f5,dish_es_v1_d5782aef2e608777,0d0946cb-a72c-4197-9d43-b5f3dbfcc620,8cb19db3-e9a8-4a42-be63-73e86e22a3d9,2acb2538-7f10-4760-a646-85c41436f9e8,a299c238d5349ed3eb5fc64b5db7e6def4db213c4ed81730074b12931536495c,ChIJ50jgAn5tEg0RBySkaJ7I8eE,Tropitali,5bd4fbf1-457b-4dcb-91f1-089ac449a5db,Casco Antiguo,e554ef0e-d9f1-4d89-8777-f92cee028f7f,SAN JULIAN,5,positive,es,Açai,acai,acai,acai,acai,internacional,internacional,clear_single,dish_candidate,dish_core,exact_alias,NaN,0.83,medium,True,eligible,dish_like_candidate,False,canonical_passthrough,19,23,"Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables.","Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables. Gracias por traer un pedaci",False,True,False,False,"Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables. Gracias por traer un pedacito de Brasil ❤️🇧🇷 volveré y seguro indicaré a mis amigos!!"
1,b70f05c16925cdb70c7f0ea3f4ce0d328d3ccc24,dish_es_v1_987dcc3d8e157f2d,0d0946cb-a72c-4197-9d43-b5f3dbfcc620,8cb19db3-e9a8-4a42-be63-73e86e22a3d9,2acb2538-7f10-4760-a646-85c41436f9e8,a299c238d5349ed3eb5fc64b5db7e6def4db213c4ed81730074b12931536495c,ChIJ50jgAn5tEg0RBySkaJ7I8eE,Tropitali,5bd4fbf1-457b-4dcb-91f1-089ac449a5db,Casco Antiguo,e554ef0e-d9f1-4d89-8777-f92cee028f7f,SAN JULIAN,5,positive,es,coxinha,coxinha,coxinha,coxinha,coxinha,internacional,internacional,clear_single,dish_candidate,dish_core,exact_alias,NaN,0.83,medium,True,eligible,dish_like_candidate,False,canonical_passthrough,59,66,"Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables.","Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables. Gracias por traer un pedacito de Brasil ❤️🇧🇷 volveré y seguro indicaré",False,True,False,False,"Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables. Gracias por traer un pedacito de Brasil ❤️🇧🇷 volveré y seguro indicaré a mis amigos!!"
2,159fb380f51c372ca4a28498d1737f9b3564b0b3,dish_es_v1_aca2ea7ae1f740a5,944c8719-53fd-49a0-9da4-4223da24a03d,a6ba370c-eb3d-4456-94cf-58617d1a3b0a,9f4c2eb7-2ff7-44ce-8db2-1be259e6985d,80289e938d35629e44007e651599356e66a6e7a106614bdae06a356afc22b758,ChIJuQj13EFvEg0RPkUdEUzL0BA,Cariña VZ,22b433e4-3b14-4b09-bfaa-f0ca1d7e6bdc,Sur,65af0982-2885-49df-9f09-8b784c2f3d7b,GIRALDA SUR,5,positive,es,arepa,arepa,arepa,arepa,arepa,internacional,internacional,clear_single,dish_candidate,dish_core,exact_alias,NaN,0.83,medium,True,eligible,dish_like_candidate,False,canonical_passthrough,102,107,"Los sabores son auténticos, la arepa de reina pepiada es divina, y el ambiente te transporta a Venezuela.","Visité este restaurante venezolano y fue una experiencia espectacular. Los sabores son auténticos, la arepa de reina pepiada es divina, y el ambiente te transporta a Venezuela. Leonardo, fue súper atento, nos recomendó platos incr

IDs únicos mention_id: True
Reviews únicas: 1452
Locales únicos: 635
Platos únicos: 181
Barrios únicos: 94
Distritos únicos: 11


In [5]:
# ============================================================
# 05. Normalización textual y utilidades regex
# ============================================================

def strip_accents(text: str) -> str:
    text = unicodedata.normalize("NFD", text)
    return "".join(ch for ch in text if unicodedata.category(ch) != "Mn")


def normalize_text_for_matching(text: Any) -> str:
    if text is None:
        return ""
    text = str(text).lower()
    text = strip_accents(text)
    text = text.replace("ñ", "n")
    text = re.sub(r"[^a-z0-9€%]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def phrase_to_regex(phrase: str) -> str:
    normalized = normalize_text_for_matching(phrase)
    return re.escape(normalized).replace("\\ ", r"\s+")


def compile_phrase_pattern(phrase: str) -> re.Pattern:
    return re.compile(rf"(?<!\w){phrase_to_regex(phrase)}(?!\w)", flags=re.IGNORECASE)


def tokenize_norm(text: str) -> list[str]:
    return re.findall(r"\b[a-z0-9]+\b", normalize_text_for_matching(text))


def clean_list(values: Iterable[Any]) -> list[str]:
    out = []
    for value in values:
        if value is None:
            continue
        norm = normalize_text_for_matching(value)
        if norm:
            out.append(norm)
    return sorted(set(out), key=lambda x: (len(x.split()), len(x)), reverse=True)


## 06. Lexicones de sentimiento en español

Los lexicones se centran en reseñas gastronómicas. Se separan términos positivos, negativos, neutros/ambiguos, negaciones, intensificadores y marcadores de contraste.

El objetivo no es crear un análisis lingüístico perfecto, sino una señal trazable y útil para agregación posterior.

In [6]:
# ============================================================
# 06. Lexicones de sentimiento gastronómico en español
# ============================================================

POSITIVE_LEXICON: dict[str, float] = {
    # positivos fuertes
    "espectacular": 2.2,
    "increible": 2.0,
    "impresionante": 1.9,
    "exquisito": 2.1,
    "exquisita": 2.1,
    "delicioso": 2.0,
    "deliciosa": 2.0,
    "riquisimo": 2.0,
    "riquisima": 2.0,
    "buenisimo": 1.9,
    "buenisima": 1.9,
    "brutal": 1.9,
    "maravilloso": 1.8,
    "maravillosa": 1.8,
    "excelente": 1.8,
    "perfecto": 1.7,
    "perfecta": 1.7,
    "de 10": 2.1,
    "top": 1.6,
    "magnifico": 1.8,
    "magnifica": 1.8,
    "fantastico": 1.9,
    "fantastica": 1.9,
    # positivos medios
    "rico": 1.4,
    "rica": 1.4,
    "sabroso": 1.4,
    "sabrosa": 1.4,
    "bueno": 1.2,
    "buena": 1.2,
    "buen": 1.0,
    "recomendable": 1.3,
    "recomiendo": 1.3,
    "recomendamos": 1.3,
    "merece la pena": 1.5,
    "merecen la pena": 1.5,
    "me encanto": 1.8,
    "nos encanto": 1.8,
    "me encantaron": 1.8,
    "nos encantaron": 1.8,
    "me gusto": 1.2,
    "nos gusto": 1.2,
    "volveria": 1.2,
    "repetiria": 1.3,
    "para repetir": 1.4,
    # atributos gastronómicos positivos
    "casero": 1.1,
    "casera": 1.1,
    "fresco": 1.1,
    "fresca": 1.1,
    "jugoso": 1.2,
    "jugosa": 1.2,
    "tierno": 1.2,
    "tierna": 1.2,
    "crujiente": 1.1,
    "cremoso": 1.1,
    "cremosa": 1.1,
    "meloso": 1.1,
    "melosa": 1.1,
    "bien hecho": 1.1,
    "bien hecha": 1.1,
    "en su punto": 1.4,
    "saborazo": 1.7,
    "sabor": 0.5,
    "buen sabor": 1.4,
    "muy buen sabor": 1.7,
    "calidad": 0.7,
    "buena calidad": 1.3,
    "alta calidad": 1.4,
    "abundante": 0.9,
    "generoso": 0.9,
    "generosa": 0.9,
}

NEGATIVE_LEXICON: dict[str, float] = {
    # negativos fuertes
    "horrible": 2.3,
    "pesimo": 2.2,
    "pesima": 2.2,
    "malisimo": 2.2,
    "malisima": 2.2,
    "asqueroso": 2.4,
    "asquerosa": 2.4,
    "incomible": 2.4,
    "decepcionante": 1.9,
    "decepcion": 1.6,
    "fatal": 2.1,
    "nunca mas": 2.2,
    "no volveria": 1.9,
    "no repetiria": 1.9,
    "no merece la pena": 2.0,
    "no valen la pena": 2.0,
    "no vale la pena": 2.0,
    # negativos de comida
    "frio": 1.7,
    "fria": 1.7,
    "seco": 1.7,
    "seca": 1.7,
    "duro": 1.5,
    "dura": 1.5,
    "quemado": 1.7,
    "quemada": 1.7,
    "crudo": 1.8,
    "cruda": 1.8,
    "pasado": 1.5,
    "pasada": 1.5,
    "salado": 1.2,
    "salada": 1.2,
    "soso": 1.4,
    "sosa": 1.4,
    "insipido": 1.7,
    "insipida": 1.7,
    "sin sabor": 1.8,
    "aceitoso": 1.5,
    "aceitosa": 1.5,
    "grasiento": 1.5,
    "grasienta": 1.5,
    "recalentado": 1.6,
    "recalentada": 1.6,
    "congelado": 1.6,
    "congelada": 1.6,
    "escaso": 1.2,
    "escasa": 1.2,
    "poca cantidad": 1.2,
    "caro": 1.0,
    "cara": 1.0,
    "carisimo": 1.5,
    "carisima": 1.5,
    "regular": 0.9,
    "normalito": 0.9,
    "normalita": 0.9,
    "flojo": 1.1,
    "floja": 1.1,
    "mejorable": 1.0,
}

NEUTRAL_OR_AMBIGUOUS_TERMS: dict[str, float] = {
    "normal": 0.6,
    "correcto": 0.5,
    "correcta": 0.5,
    "aceptable": 0.5,
    "sin mas": 0.7,
    "esta bien": 0.4,
    "bien sin mas": 0.7,
    "normalito": 0.8,
    "normalita": 0.8,
}

NEGATION_TERMS = {
    "no", "nunca", "jamas", "tampoco", "ni", "sin", "nada", "apenas", "poco", "poca", "ningun", "ninguna"
}

INTENSIFIERS = {
    "muy", "super", "súper", "bastante", "realmente", "totalmente", "absolutamente", "increiblemente", "demasiado", "tan"
}

CONTRAST_MARKERS = {
    "pero", "aunque", "sin embargo", "no obstante", "aun asi", "eso si", "en cambio", "mientras que"
}

PRICE_TERMS = {"caro", "cara", "carisimo", "carisima", "precio", "precios", "coste", "costaba", "vale", "valia"}
SERVICE_TERMS = {"servicio", "camarero", "camarera", "camareros", "camareras", "atencion", "trato", "personal", "mesa", "espera"}

POSITIVE_PATTERNS = [(term, weight, compile_phrase_pattern(term)) for term, weight in POSITIVE_LEXICON.items()]
NEGATIVE_PATTERNS = [(term, weight, compile_phrase_pattern(term)) for term, weight in NEGATIVE_LEXICON.items()]
NEUTRAL_PATTERNS = [(term, weight, compile_phrase_pattern(term)) for term, weight in NEUTRAL_OR_AMBIGUOUS_TERMS.items()]
CONTRAST_PATTERNS = [(term, compile_phrase_pattern(term)) for term in CONTRAST_MARKERS]

print("Términos positivos:", len(POSITIVE_PATTERNS))
print("Términos negativos:", len(NEGATIVE_PATTERNS))
print("Términos neutros/ambiguos:", len(NEUTRAL_PATTERNS))


Términos positivos: 70
Términos negativos: 57
Términos neutros/ambiguos: 9


In [7]:
# ============================================================
# 07. Construcción de contexto local por mención
# ============================================================

def first_non_empty(*values: Any) -> str:
    for value in values:
        if value is None:
            continue
        text = str(value).strip()
        if text and text.lower() != "nan":
            return text
    return ""


def find_mention_index(context_norm: str, mention_norm: str, canonical_norm: str) -> int:
    if mention_norm:
        idx = context_norm.find(mention_norm)
        if idx >= 0:
            return idx
    if canonical_norm:
        idx = context_norm.find(canonical_norm)
        if idx >= 0:
            return idx
    return -1


def build_target_context(row: pd.Series, max_chars: int = 260) -> dict[str, Any]:
    """Devuelve contexto usado para sentimiento y metadatos de localización de la mención."""
    context_sentence = first_non_empty(row.get("context_sentence"), row.get("window_context"), row.get("text"))
    window_context = first_non_empty(row.get("window_context"), context_sentence, row.get("text"))
    full_text = first_non_empty(row.get("text"), window_context, context_sentence)

    base_context = context_sentence if len(context_sentence) >= 10 else window_context
    if len(base_context) < 20 and full_text:
        base_context = full_text

    base_norm = normalize_text_for_matching(base_context)
    mention_norm = normalize_text_for_matching(row.get("dish_mention_text"))
    canonical_norm = normalize_text_for_matching(row.get("display_dish_name_es_v1") or row.get("normalized_dish_name_es_v1"))

    idx = find_mention_index(base_norm, mention_norm, canonical_norm)

    # Si el contexto es largo, recortamos alrededor de la primera aparición normalizada.
    if len(base_context) > max_chars and idx >= 0:
        # Aproximación: idx en texto normalizado. Para recorte visual usamos proporción.
        ratio = idx / max(len(base_norm), 1)
        approx_char = int(ratio * len(base_context))
        start = max(0, approx_char - max_chars // 2)
        end = min(len(base_context), approx_char + max_chars // 2)
        target_context = base_context[start:end].strip()
    elif len(base_context) > max_chars:
        target_context = base_context[:max_chars].strip()
    else:
        target_context = base_context.strip()

    target_norm = normalize_text_for_matching(target_context)
    target_idx = find_mention_index(target_norm, mention_norm, canonical_norm)

    return {
        "target_context": target_context,
        "target_context_normalized": target_norm,
        "mention_norm": mention_norm,
        "canonical_norm": canonical_norm,
        "mention_index_normalized": target_idx,
        "context_source": "context_sentence" if context_sentence else "window_context_or_text",
    }

# Prueba rápida sobre algunas menciones
sample_contexts = df.head(5).apply(build_target_context, axis=1).tolist()
sample_contexts[:2]


[{'target_context': 'Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables.',
  'target_context_normalized': 'sin dudas el mejor acai que he probado aqui en sevilla la coxinha muy rica y el trato del personal increible son muy muy muy amables',
  'mention_norm': 'acai',
  'canonical_norm': 'acai',
  'mention_index_normalized': 19,
  'context_source': 'context_sentence'},
 {'target_context': 'Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables.',
  'target_context_normalized': 'sin dudas el mejor acai que he probado aqui en sevilla la coxinha muy rica y el trato del personal increible son muy muy muy amables',
  'mention_norm': 'coxinha',
  'canonical_norm': 'coxinha',
  'mention_index_normalized': 58,
  'context_source': 'context_sentence'}]

In [8]:
# ============================================================
# 08. Funciones de scoring local
# ============================================================

def has_negation_before(text_norm: str, match_start: int, window_chars: int = 32) -> bool:
    before = text_norm[max(0, match_start - window_chars):match_start]
    tokens = set(re.findall(r"\b[a-z0-9]+\b", before))
    return bool(tokens & NEGATION_TERMS)


def has_intensifier_before(text_norm: str, match_start: int, window_chars: int = 28) -> bool:
    before = text_norm[max(0, match_start - window_chars):match_start]
    tokens = set(re.findall(r"\b[a-z0-9]+\b", before))
    normalized_intensifiers = {normalize_text_for_matching(x) for x in INTENSIFIERS}
    return bool(tokens & normalized_intensifiers)


def proximity_multiplier(match_center: int, mention_index: int, mention_len: int) -> float:
    if mention_index < 0:
        return 0.95
    mention_center = mention_index + max(mention_len, 1) / 2
    distance = abs(match_center - mention_center)
    if distance <= 45:
        return 1.18
    if distance <= 90:
        return 1.0
    if distance <= 150:
        return 0.82
    return 0.62


def detect_contrast(text_norm: str) -> bool:
    return any(pattern.search(text_norm) for _, pattern in CONTRAST_PATTERNS)


def detect_service_context_risk(text_norm: str, positive_terms: list[str], negative_terms: list[str]) -> bool:
    tokens = set(tokenize_norm(text_norm))
    has_service = bool(tokens & SERVICE_TERMS)
    has_sentiment = bool(positive_terms or negative_terms)
    return has_service and has_sentiment


def detect_price_context(text_norm: str, matched_terms: list[str]) -> bool:
    tokens = set(tokenize_norm(text_norm))
    matched_norms = {normalize_text_for_matching(x.split(":", 1)[0]) for x in matched_terms}
    return bool((tokens | matched_norms) & PRICE_TERMS)


def find_weighted_terms(
    text_norm: str,
    patterns: list[tuple[str, float, re.Pattern]],
    mention_index: int,
    mention_len: int,
    polarity: str,
) -> tuple[float, float, list[str], list[str], list[str]]:
    """Devuelve pos_score_delta, neg_score_delta, terms, flags, debug matches."""
    pos_delta = 0.0
    neg_delta = 0.0
    terms: list[str] = []
    flags: list[str] = []
    debug_matches: list[str] = []

    for term, base_weight, pattern in patterns:
        for match in pattern.finditer(text_norm):
            match_center = (match.start() + match.end()) / 2
            prox = proximity_multiplier(match_center, mention_index, mention_len)
            weight = base_weight * prox

            if has_intensifier_before(text_norm, match.start()):
                weight *= 1.18
                flags.append("intensifier_nearby")

            negated = has_negation_before(text_norm, match.start())

            if polarity == "positive":
                if negated:
                    neg_delta += weight * 0.85
                    terms.append(f"{term}:negated_positive")
                    flags.append("negation_detected")
                else:
                    pos_delta += weight
                    terms.append(f"{term}:positive_lexicon")
            else:
                if negated:
                    # "no estaba malo" no siempre es claramente positivo. Lo tratamos como señal positiva débil.
                    pos_delta += weight * 0.35
                    terms.append(f"{term}:negated_negative")
                    flags.append("negation_detected")
                else:
                    neg_delta += weight
                    terms.append(f"{term}:negative_lexicon")

            debug_matches.append(f"{term}@{match.start()}:{weight:.2f}")

    return pos_delta, neg_delta, terms, flags, debug_matches


def find_neutral_terms(text_norm: str) -> tuple[float, list[str]]:
    neutral_score = 0.0
    terms: list[str] = []
    for term, weight, pattern in NEUTRAL_PATTERNS:
        if pattern.search(text_norm):
            neutral_score += weight
            terms.append(f"{term}:neutral_or_ambiguous")
    return neutral_score, terms


In [9]:
# ============================================================
# 09. Fallback desde rating y decisión final de etiqueta
# ============================================================

def rating_prior(rating_value: Any) -> tuple[str, float, float]:
    """Devuelve label, score y confianza base a partir del rating."""
    try:
        rating = float(rating_value)
    except (TypeError, ValueError):
        return "neutral", 0.0, 0.25

    if rating >= 4.5:
        return "positive", 1.15, 0.42
    if rating >= 4.0:
        return "positive", 0.85, 0.38
    if rating >= 3.0:
        return "neutral", 0.0, 0.34
    if rating >= 2.0:
        return "negative", -0.85, 0.38
    return "negative", -1.15, 0.42


def label_from_local_score(score: float, neutral_score: float) -> str:
    # Si hay términos neutros y el score polar es pequeño, forzamos neutralidad.
    if neutral_score >= 0.6 and abs(score) < 1.0:
        return "neutral"
    if score >= 0.55:
        return "positive"
    if score <= -0.55:
        return "negative"
    return "neutral"


def compute_confidence(
    reason: str,
    label: str,
    local_strength: float,
    score: float,
    neutral_score: float,
    rating_label: str,
    flags: list[str],
) -> float:
    if reason == "local_context_primary":
        conf = 0.52 + min(local_strength / 4.0, 0.34)
        if abs(score) >= 2.0:
            conf += 0.05
        if label == rating_label and label != "neutral":
            conf += 0.04
        if "mixed_local_evidence" in flags:
            conf -= 0.08
        if "contrast_marker_nearby" in flags:
            conf -= 0.04
        if "sentiment_may_refer_to_service" in flags:
            conf -= 0.07
        if "price_context_detected" in flags:
            conf -= 0.03
    else:
        conf = 0.30
        if rating_label in {"positive", "negative"}:
            conf += 0.08
        if neutral_score > 0:
            conf += 0.02
        if "weak_local_sentiment_signal" in flags:
            conf -= 0.02

    return float(max(0.18, min(0.95, round(conf, 4))))


def sentiment_weight(reason: str, confidence: float) -> float:
    """Peso sugerido para agregación posterior."""
    if reason == "local_context_primary":
        return round(confidence, 4)
    return round(confidence * 0.62, 4)


In [10]:
# ============================================================
# 10. Función principal de sentimiento por mención
# ============================================================

def compute_mention_sentiment(row: pd.Series) -> dict[str, Any]:
    context_info = build_target_context(row)
    text_norm = context_info["target_context_normalized"]
    mention_index = context_info["mention_index_normalized"]
    mention_len = len(context_info["mention_norm"] or context_info["canonical_norm"])

    pos_score = 0.0
    neg_score = 0.0
    flags: list[str] = []
    positive_terms: list[str] = []
    negative_terms: list[str] = []
    debug_matches: list[str] = []

    p_pos, p_neg, p_terms, p_flags, p_debug = find_weighted_terms(
        text_norm, POSITIVE_PATTERNS, mention_index, mention_len, polarity="positive"
    )
    n_pos, n_neg, n_terms, n_flags, n_debug = find_weighted_terms(
        text_norm, NEGATIVE_PATTERNS, mention_index, mention_len, polarity="negative"
    )

    pos_score += p_pos + n_pos
    neg_score += p_neg + n_neg
    positive_terms.extend([t for t in p_terms + n_terms if "positive" in t or "negated_negative" in t])
    negative_terms.extend([t for t in p_terms + n_terms if "negative" in t or "negated_positive" in t])
    flags.extend(p_flags + n_flags)
    debug_matches.extend(p_debug + n_debug)

    neutral_score, neutral_terms = find_neutral_terms(text_norm)

    if detect_contrast(text_norm):
        flags.append("contrast_marker_nearby")
    if pos_score > 0 and neg_score > 0:
        flags.append("mixed_local_evidence")
    if detect_service_context_risk(text_norm, positive_terms, negative_terms):
        flags.append("sentiment_may_refer_to_service")
    if detect_price_context(text_norm, positive_terms + negative_terms):
        flags.append("price_context_detected")
    if neutral_terms:
        flags.append("neutral_or_ambiguous_terms_detected")

    local_score = pos_score - neg_score
    local_strength = pos_score + neg_score

    rating_label, rating_score, rating_conf = rating_prior(row.get("rating_value"))

    if local_strength >= 0.45 or neutral_score >= 0.6:
        label = label_from_local_score(local_score, neutral_score)
        reason = "local_context_primary"
        final_score = local_score
    else:
        label = rating_label
        reason = "review_prior_fallback"
        final_score = rating_score
        flags.append("used_review_prior_fallback")
        flags.append("weak_local_sentiment_signal")

    if local_strength < 0.45 and neutral_score < 0.6:
        flags.append("no_clear_local_sentiment")

    confidence = compute_confidence(
        reason=reason,
        label=label,
        local_strength=local_strength,
        score=local_score,
        neutral_score=neutral_score,
        rating_label=rating_label,
        flags=list(set(flags)),
    )

    if confidence < 0.4:
        flags.append("low_sentiment_confidence")

    return {
        "mention_sentiment_label_v1": label,
        "mention_sentiment_score_v1": round(float(final_score), 4),
        "mention_sentiment_local_score_v1": round(float(local_score), 4),
        "mention_sentiment_local_strength_v1": round(float(local_strength), 4),
        "mention_sentiment_confidence_v1": confidence,
        "mention_sentiment_weight_v1": sentiment_weight(reason, confidence),
        "sentiment_reason_v1": reason,
        "sentiment_flags_v1": sorted(set(flags)),
        "positive_terms_v1": sorted(set(positive_terms)),
        "negative_terms_v1": sorted(set(negative_terms)),
        "neutral_terms_v1": sorted(set(neutral_terms)),
        "rating_prior_label_v1": rating_label,
        "rating_prior_score_v1": round(float(rating_score), 4),
        "context_used_v1": context_info["target_context"],
        "context_source_v1": context_info["context_source"],
        "debug_sentiment_matches_v1": sorted(set(debug_matches)),
    }

# Smoke test manual
manual_examples = pd.DataFrame([
    {"dish_mention_text": "croquetas", "display_dish_name_es_v1": "croqueta", "context_sentence": "Las croquetas estaban espectaculares y muy cremosas.", "window_context": "", "text": "", "rating_value": 5},
    {"dish_mention_text": "tortilla", "display_dish_name_es_v1": "tortilla de patatas", "context_sentence": "La tortilla estaba fría y bastante seca.", "window_context": "", "text": "", "rating_value": 2},
    {"dish_mention_text": "ensaladilla", "display_dish_name_es_v1": "ensaladilla", "context_sentence": "Pedimos ensaladilla, croquetas y montaditos.", "window_context": "", "text": "", "rating_value": 4},
])

for _, ex in manual_examples.iterrows():
    print(ex["context_sentence"])
    print(compute_mention_sentiment(ex))
    print("-" * 80)


Las croquetas estaban espectaculares y muy cremosas.
{'mention_sentiment_label_v1': 'positive', 'mention_sentiment_score_v1': 1.15, 'mention_sentiment_local_score_v1': 0.0, 'mention_sentiment_local_strength_v1': 0.0, 'mention_sentiment_confidence_v1': 0.36, 'mention_sentiment_weight_v1': 0.2232, 'sentiment_reason_v1': 'review_prior_fallback', 'sentiment_flags_v1': ['low_sentiment_confidence', 'no_clear_local_sentiment', 'used_review_prior_fallback', 'weak_local_sentiment_signal'], 'positive_terms_v1': [], 'negative_terms_v1': [], 'neutral_terms_v1': [], 'rating_prior_label_v1': 'positive', 'rating_prior_score_v1': 1.15, 'context_used_v1': 'Las croquetas estaban espectaculares y muy cremosas.', 'context_source_v1': 'context_sentence', 'debug_sentiment_matches_v1': []}
--------------------------------------------------------------------------------
La tortilla estaba fría y bastante seca.
{'mention_sentiment_label_v1': 'negative', 'mention_sentiment_score_v1': -4.3731, 'mention_sentiment

In [11]:
# ============================================================
# 11. Aplicar sentimiento a todas las menciones ranking-ready
# ============================================================

sentiment_rows = df.apply(compute_mention_sentiment, axis=1, result_type="expand")
mentions_sentiment_df = pd.concat([df.reset_index(drop=True), sentiment_rows.reset_index(drop=True)], axis=1)

print("Menciones con sentimiento:", len(mentions_sentiment_df))
print("Distribución de sentimiento:")
display(mentions_sentiment_df["mention_sentiment_label_v1"].value_counts())

print("Distribución de reason:")
display(mentions_sentiment_df["sentiment_reason_v1"].value_counts())

print("Confianza:")
display(mentions_sentiment_df["mention_sentiment_confidence_v1"].describe())


Menciones con sentimiento: 2979
Distribución de sentimiento:


mention_sentiment_label_v1
positive    2395
negative     388
neutral      196
Name: count, dtype: int64

Distribución de reason:


sentiment_reason_v1
local_context_primary    1590
review_prior_fallback    1389
Name: count, dtype: int64

Confianza:


count    2979.000000
mean        0.631865
std         0.266814
min         0.280000
25%         0.360000
50%         0.710000
75%         0.900000
max         0.950000
Name: mention_sentiment_confidence_v1, dtype: float64

In [12]:
# ============================================================
# 12. Diagnósticos cruzados
# ============================================================

rating_sentiment_crosstab = pd.crosstab(
    mentions_sentiment_df["rating_value"],
    mentions_sentiment_df["mention_sentiment_label_v1"],
    margins=True,
)

general_review_sentiment_crosstab = pd.crosstab(
    mentions_sentiment_df["review_sentiment_from_rating"],
    mentions_sentiment_df["mention_sentiment_label_v1"],
    margins=True,
)

reason_crosstab = pd.crosstab(
    mentions_sentiment_df["sentiment_reason_v1"],
    mentions_sentiment_df["mention_sentiment_label_v1"],
    margins=True,
)

print("Rating vs sentimiento de mención:")
display(rating_sentiment_crosstab)

print("Sentimiento general de review vs sentimiento de mención:")
display(general_review_sentiment_crosstab)

print("Reason vs sentimiento:")
display(reason_crosstab)


Rating vs sentimiento de mención:


mention_sentiment_label_v1,negative,neutral,positive,All
rating_value,,,,
1,178,12,17,207
2,102,13,21,136
3,28,106,49,183
4,42,29,517,588
5,38,36,1791,1865
All,388,196,2395,2979


Sentimiento general de review vs sentimiento de mención:


mention_sentiment_label_v1,negative,neutral,positive,All
review_sentiment_from_rating,,,,
negative,280,25,38,343
neutral,28,106,49,183
positive,80,65,2308,2453
All,388,196,2395,2979


Reason vs sentimiento:


mention_sentiment_label_v1,negative,neutral,positive,All
sentiment_reason_v1,,,,
local_context_primary,192,103,1295,1590
review_prior_fallback,196,93,1100,1389
All,388,196,2395,2979


In [13]:
# ============================================================
# 13. Flags y ejemplos para revisión
# ============================================================

def explode_list_column(df_: pd.DataFrame, column: str) -> pd.Series:
    counter = Counter()
    for values in df_[column].fillna(""):
        if isinstance(values, list):
            counter.update(values)
        elif isinstance(values, str) and values:
            counter.update([values])
    return pd.Series(dict(counter)).sort_values(ascending=False)

flag_counts = explode_list_column(mentions_sentiment_df, "sentiment_flags_v1")
positive_term_counts = explode_list_column(mentions_sentiment_df, "positive_terms_v1")
negative_term_counts = explode_list_column(mentions_sentiment_df, "negative_terms_v1")

print("Flags principales:")
display(flag_counts.head(30))

print("Términos positivos principales:")
display(positive_term_counts.head(30))

print("Términos negativos principales:")
display(negative_term_counts.head(30))

manual_review_mask = (
    (mentions_sentiment_df["mention_sentiment_confidence_v1"] < 0.48)
    | mentions_sentiment_df["sentiment_flags_v1"].apply(lambda flags: "mixed_local_evidence" in flags if isinstance(flags, list) else False)
    | mentions_sentiment_df["sentiment_flags_v1"].apply(lambda flags: "sentiment_may_refer_to_service" in flags if isinstance(flags, list) else False)
    | mentions_sentiment_df["sentiment_flags_v1"].apply(lambda flags: "contrast_marker_nearby" in flags if isinstance(flags, list) else False)
)

manual_review_df = mentions_sentiment_df[manual_review_mask].copy()
manual_review_df = manual_review_df.sort_values(
    ["mention_sentiment_confidence_v1", "mention_sentiment_local_strength_v1"],
    ascending=[True, False],
)

manual_cols = [
    "mention_id", "review_id", "place_id", "place_name", "district_name", "neighborhood_name",
    "rating_value", "review_sentiment_from_rating",
    "display_dish_name_es_v1", "dish_mention_text",
    "mention_sentiment_label_v1", "mention_sentiment_score_v1", "mention_sentiment_confidence_v1",
    "sentiment_reason_v1", "sentiment_flags_v1", "positive_terms_v1", "negative_terms_v1",
    "context_used_v1", "text"
]
manual_cols = [c for c in manual_cols if c in manual_review_df.columns]

display(manual_review_df[manual_cols].head(30))
print("Menciones marcadas para revisión:", len(manual_review_df))


Flags principales:


low_sentiment_confidence               1389
used_review_prior_fallback             1389
no_clear_local_sentiment               1389
weak_local_sentiment_signal            1389
intensifier_nearby                      528
contrast_marker_nearby                  398
negation_detected                       176
sentiment_may_refer_to_service          172
mixed_local_evidence                    156
price_context_detected                  115
neutral_or_ambiguous_terms_detected      93
dtype: int64

Términos positivos principales:


buena:positive_lexicon           225
sabor:positive_lexicon           185
bueno:positive_lexicon           176
espectacular:positive_lexicon    144
recomiendo:positive_lexicon      104
rico:positive_lexicon             94
buenisimo:positive_lexicon        94
calidad:positive_lexicon          94
buen:positive_lexicon             82
rica:positive_lexicon             69
increible:positive_lexicon        58
excelente:positive_lexicon        57
recomendable:positive_lexicon     55
perfecto:positive_lexicon         48
buenisima:positive_lexicon        45
riquisima:positive_lexicon        42
deliciosa:positive_lexicon        38
sabor:negated_positive            37
delicioso:positive_lexicon        36
de 10:positive_lexicon            36
casera:positive_lexicon           35
riquisimo:positive_lexicon        33
exquisita:positive_lexicon        32
nos gusto:positive_lexicon        31
sabrosa:positive_lexicon          29
casero:positive_lexicon           28
exquisito:positive_lexicon        25
e

Términos negativos principales:


sabor:negated_positive            37
seco:negative_lexicon             19
frio:negative_lexicon             19
sin sabor:negative_lexicon        15
crudo:negative_lexicon            14
fria:negative_lexicon             13
nos gusto:negated_positive        12
cara:negative_lexicon             11
congelado:negative_lexicon        11
bueno:negated_positive            10
duro:negative_lexicon              9
regular:negative_lexicon           9
seca:negative_lexicon              9
dura:negative_lexicon              8
mejorable:negative_lexicon         8
decepcionante:negative_lexicon     8
recomiendo:negated_positive        7
pasada:negative_lexicon            7
espectacular:negated_positive      7
me gusto:negated_positive          6
buen:negated_positive              6
sosa:negated_negative              5
escasa:negative_lexicon            5
calidad:negated_positive           5
caro:negative_lexicon              5
pasado:negative_lexicon            5
escaso:negative_lexicon            5
s

,mention_id,review_id,place_id,place_name,district_name,neighborhood_name,rating_value,review_sentiment_from_rating,display_dish_name_es_v1,dish_mention_text,mention_sentiment_label_v1,mention_sentiment_score_v1,mention_sentiment_confidence_v1,sentiment_reason_v1,sentiment_flags_v1,positive_terms_v1,negative_terms_v1,context_used_v1,text
17,101dac5ea4cc81a3dfec6ff9537435e69fb75f3d,1b12f8c1-1e42-4117-b809-19e7fb58dea7,34a53dbf-ac03-4576-bf1f-0265341e9c82,Café bar de la Encarnación,Casco Antiguo,ENCARNACIÓN-REGINA,3,neutral,montadito,montadito,neutral,0.0,0.28,review_prior_fallback,"[low_sentiment_confidence, no_clear_local_sentiment, used_review_prior_fallback, weak_local_sentiment_signal]",[],[],El serranito está a medio camino del montadito por au tamaño.,"He probado el serranito. Me han cobrado un euro más de lo que tienen escrito en la carta, previo aviso, lo cual ya me han cobrado piesto mal cuerpo. El serranito está a medio camino del montadito por au tamaño. Un serranito tiene q venir en viena o amdaluz..."
47,20e112ad2091d079acace7b599d9866cc1c14f64,f5d7e4be-8328-48bc-8ec0-76542dd6f752,80915e5a-8e3b-4659-a094-3ed071a8dac5,Restaurante El Caña,Los Remedios,LOS REMEDIOS,3,neutral,bacalao,bacalao,neutral,0.0,0.28,review_prior_fallback,"[low_sentiment_confidence, no_clear_local_sentiment, used_review_prior_fallback, weak_local_sentiment_signal]",[],[],"Estuvimos 6 personas y además de tener un servicio lento el primer plato de aliño de patatas con bacalao traía el pescado en mal estado, con un olor que era imposible que el cocinero o el camarero no lo percibieran.","Estuvimos 6 personas y además de tener un servicio lento el primer plato de aliño de patatas con bacalao traía el pescado en mal estado, con un olor que era imposible que el cocinero o el camarero no lo percibieran. El resto de los platos bien, sin más. Pe..."
173,10c6a1dc15027f2a7e37f088c689d0128ea3bbf3,96f09344-b0c9-4c9b-bb8f-fae033fa1f94,e6aeb47b-bc1c-4306-b7a6-4ed3bb26b81d,Restaurante flores gourmet,Casco Antiguo,MUSEO,3,neutral,presa ibérica,presa iberica,neutral,0.0,0.28,review_prior_fallback,"[low_sentiment_confidence, no_clear_local_sentiment, used_review_prior_fallback, weak_local_sentiment_signal]",[],[],El morcón de presa iberica de bellota como dice en la carta no es un morcon choricero como vino en el plato.,"Cuando pruebas productos más que aceptables y de repente te sirven algo que ni se parece a lo que has pedido en lo que a calidad se refiere, te llevas una gran decepción porque piensas o que te intentan engañar (pienso que no) o que no hablamos el mismo id..."
175,a5fa86e86e4a27d0cadc987c16cca72f1e9ee2f2,4924c76e-9466-4623-b3d7-d3b27f556d9d,85884689-c71a-4605-974e-f103cb01c1e3,Antigüedades Bar de Tapas,Casco Antiguo,SANTA CRUZ,3,neutral,tortilla de patatas,tortilla,neutral,0.0,0.28,review_prior_fallback,"[low_sentiment_confidence, no_clear_local_sentiment, used_review_prior_fallback, weak_local_sentiment_signal]",[],[],"Nos pedimos una tortilla, unas carrilleras y unas croquetas de jamón.","Nos pedimos una tortilla, unas carrilleras y unas croquetas de jamón. La tortilla es la peor que probé en mis 10 años en España, es seca como lengua de loro, le falta sal y ese salmorejo que le ponen arriba no suma, más bien resta. Muy solida y las patatas..."
176,4c6bbab31571997a6dd6b0592a69a0c14d1908d2,4924c76e-9466-4623-b3d7-d3b27f556d9d,85884689-c71a-4605-974e-f103cb01c1e3,Antigüedades Bar de Tapas,Casco Antiguo,SANTA CRUZ,3,neutral,carrillada,carrilleras,neutral,0.0,0.28,review_prior_fallback,"[low_sentiment_confidence, no_clear_local_sentiment, used_review_prior_fallback, weak_local_sentiment_signal]",[],[],"Nos pedimos una tortilla, unas carrilleras y unas croquetas de jamón.","Nos pedimos una tortilla, unas carrilleras y unas croquetas de jamón. La tortilla es la peor que probé en mis 10 años en España, es seca como lengua de loro, le falta sal y ese salmorejo que le ponen arriba no suma, más bien resta. Muy solida y las patatas..."
177,817bc8d5b38226

Menciones marcadas para revisión: 1896


In [18]:
# ============================================================
# 14. Resúmenes por plato, distrito, barrio y local
# ============================================================

def safe_nunique(group: pd.DataFrame, column: str, fallback: int = 0) -> int:
    """
    Devuelve nunique de una columna si existe en el grupo.
    Si la columna no está disponible porque forma parte del groupby,
    devuelve un fallback seguro.
    """
    if column in group.columns:
        return int(group[column].nunique(dropna=True))
    return fallback


def safe_mean(group: pd.DataFrame, column: str, fallback: float = 0.0) -> float:
    """
    Devuelve media de una columna si existe y tiene valores.
    """
    if column in group.columns and len(group[column]) > 0:
        return round(float(group[column].mean()), 4)
    return fallback


def sentiment_agg(group: pd.DataFrame) -> pd.Series:
    total = len(group)

    pos = int((group["mention_sentiment_label_v1"] == "positive").sum())
    neu = int((group["mention_sentiment_label_v1"] == "neutral").sum())
    neg = int((group["mention_sentiment_label_v1"] == "negative").sum())

    if total:
        weights = np.maximum(group["mention_sentiment_weight_v1"].fillna(0.01), 0.01)
        weighted_score = np.average(
            group["mention_sentiment_score_v1"].fillna(0.0),
            weights=weights,
        )
    else:
        weighted_score = 0.0

    return pd.Series({
        "mention_count": total,
        "review_count": safe_nunique(group, "review_id", fallback=total),
        # Si place_id no está dentro del grupo porque es clave del groupby,
        # en un resumen place-dish el número de locales del grupo es 1.
        "place_count": safe_nunique(group, "place_id", fallback=1 if total else 0),
        "positive_count": pos,
        "neutral_count": neu,
        "negative_count": neg,
        "positive_ratio": round(pos / total, 4) if total else 0.0,
        "neutral_ratio": round(neu / total, 4) if total else 0.0,
        "negative_ratio": round(neg / total, 4) if total else 0.0,
        "avg_sentiment_score": safe_mean(group, "mention_sentiment_score_v1"),
        "weighted_sentiment_score": round(float(weighted_score), 4),
        "avg_sentiment_confidence": safe_mean(group, "mention_sentiment_confidence_v1"),
        "local_context_ratio": round(
            float((group["sentiment_reason_v1"] == "local_context_primary").mean()),
            4
        ) if "sentiment_reason_v1" in group.columns and total else 0.0,
        "avg_rating": safe_mean(group, "rating_value"),
    })


summary_by_dish = (
    mentions_sentiment_df
    .groupby(
        [
            "dish_id_v1",
            "display_dish_name_es_v1",
            "normalized_dish_name_es_v1",
            "dish_family_es_v1",
            "dish_group_es_v1",
        ],
        dropna=False
    )
    .apply(sentiment_agg)
    .reset_index()
    .sort_values(
        ["mention_count", "positive_ratio", "weighted_sentiment_score"],
        ascending=[False, False, False]
    )
)

summary_by_district = (
    mentions_sentiment_df
    .groupby(["district_id", "district_name"], dropna=False)
    .apply(sentiment_agg)
    .reset_index()
    .sort_values(
        ["mention_count", "weighted_sentiment_score"],
        ascending=[False, False]
    )
)

summary_by_neighborhood = (
    mentions_sentiment_df
    .groupby(["district_name", "neighborhood_id", "neighborhood_name"], dropna=False)
    .apply(sentiment_agg)
    .reset_index()
    .sort_values(
        ["mention_count", "weighted_sentiment_score"],
        ascending=[False, False]
    )
)

summary_by_place_dish = (
    mentions_sentiment_df
    .groupby(
        [
            "place_id",
            "place_name",
            "district_name",
            "neighborhood_name",
            "dish_id_v1",
            "display_dish_name_es_v1",
            "normalized_dish_name_es_v1",
        ],
        dropna=False
    )
    .apply(sentiment_agg)
    .reset_index()
    .sort_values(
        ["mention_count", "positive_ratio", "weighted_sentiment_score"],
        ascending=[False, False, False]
    )
)

print("Resumen por plato:")
display(summary_by_dish.head(30))

print("Resumen por local-plato:")
display(summary_by_place_dish.head(30))

Resumen por plato:


,dish_id_v1,display_dish_name_es_v1,normalized_dish_name_es_v1,dish_family_es_v1,dish_group_es_v1,mention_count,review_count,place_count,positive_count,neutral_count,negative_count,positive_ratio,neutral_ratio,negative_ratio,avg_sentiment_score,weighted_sentiment_score,avg_sentiment_confidence,local_context_ratio,avg_rating
13,dish_es_v1_10850a8a8e314902,ensaladilla,ensaladilla,tapas_clasicas,ensaladilla,216.0,197.0,153.0,167.0,18.0,31.0,0.7731,0.0833,0.1435,1.3194,1.7808,0.6275,0.5324,4.1806
39,dish_es_v1_3b8c4e72931fd3d2,croqueta,croqueta,tapas_clasicas,croqueta,211.0,198.0,160.0,166.0,17.0,28.0,0.7867,0.0806,0.1327,1.1589,1.5156,0.6072,0.4882,4.3128
32,dish_es_v1_2c67f67ce386e9d6,carrillada,carrillada,tapas_clasicas,carrillada,172.0,152.0,123.0,149.0,12.0,11.0,0.8663,0.0698,0.0640,1.6571,2.1338,0.6390,0.5407,4.4884
9,dish_es_v1_0a420a9251bb5aed,solomillo al whisky,solomillo al whisky,tapas_clasicas,solomillo,167.0,152.0,126.0,132.0,10.0,25.0,0.7904,0.0599,0.1497,1.3752,1.8203,0.6495,0.5689,4.2096
114,dish_es_v1_ad3755cf6fd0fb67,atún,atun,mar_y_pescado,pescado,117.0,99.0,70.0,94.0,8.0,15.0,0.8034,0.0684,0.1282,1.5184,1.8470,0.7066,0.6752,4.4017
21,dish_es_v1_18ef91885a7312e4,tarta de queso,tarta de queso,postres_y_desayunos,tarta,114.0,100.0,81.0,96.0,9.0,9.0,0.8421,0.0789,0.0789,1.4979,1.9128,0.6626,0.5702,4.4737
105,dish_es_v1_a646e68b8a85d929,montadito,montadito,tapas_clasicas,montadito,108.0,95.0,73.0,87.0,6.0,15.0,0.8056,0.0556,0.1389,1.1354,1.5037,0.5647,0.3981,4.3981
136,dish_es_v1_c325d21226b882b6,patatas bravas,patatas bravas,fritos_y_raciones,patatas,105.0,98.0,76.0,83.0,7.0,15.0,0.7905,0.0667,0.1429,1.0934,1.4183,0.5900,0.4667,4.1619
3,dish_es_v1_03d0d732fdd6fe23,gambas,gambas,mar_y_pescado,pescado,104.0,97.0,79.0,80.0,12.0,12.0,0.7692,0.1154,0.1154,1.3168,1.8086,0.6060,0.5192,4.1635
96,dish_es_v1_9a1e795ae0c1d04f,hamburguesa,hamburguesa,carne,carne,90.0,75.0,51.0,63.0,8.0,19.0,0.7000,0.0889,0.2111,0.9992,1.3733,0.5821,0.4556,3.9667


Resumen por local-plato:


,place_id,place_name,district_name,neighborhood_name,dish_id_v1,display_dish_name_es_v1,normalized_dish_name_es_v1,mention_count,review_count,place_count,positive_count,neutral_count,negative_count,positive_ratio,neutral_ratio,negative_ratio,avg_sentiment_score,weighted_sentiment_score,avg_sentiment_confidence,local_context_ratio,avg_rating
675,4d74b42b-54ef-4d31-9379-7df53dd7f6f6,Bendito Placer,Cerro - Amate,EL CERRO,dish_es_v1_9a1e795ae0c1d04f,hamburguesa,hamburguesa,10.0,5.0,1.0,5.0,1.0,4.0,0.5000,0.1000,0.4000,-0.3905,-1.1626,0.5956,0.5000,3.2000
1410,a884a37e-3e47-49b8-ba80-83ce58ce27b6,LA TENTACIÓN BURGUER,Norte,SAN DIEGO,dish_es_v1_9a1e795ae0c1d04f,hamburguesa,hamburguesa,9.0,5.0,1.0,8.0,1.0,0.0,0.8889,0.1111,0.0000,0.9908,1.1268,0.5006,0.3333,4.4444
1321,9b6cc82a-5140-4158-98e4-0b2110ec92bf,Il Ristorantino Dell´Avvocato Sevilla,Casco Antiguo,ENCARNACIÓN-REGINA,dish_es_v1_1f6ccd2be75f1cc9,pizza,pizza,9.0,5.0,1.0,7.0,2.0,0.0,0.7778,0.2222,0.0000,1.6851,2.3357,0.6578,0.5556,4.2222
1038,775735df-c2ed-4908-93c6-cc31da992abb,Heladería Valentina,Nervión,HUERTA DEL PILAR,dish_es_v1_d35dbc1062811528,helado,helado,9.0,4.0,1.0,5.0,0.0,4.0,0.5556,0.0000,0.4444,0.0168,-0.5801,0.6170,0.5556,3.2222
970,6ee0277a-c373-4d0d-94bc-f7d90766cdc2,I Love Churros - Cafetería churreria en Ronda de Capuchinos,Macarena,CRUZ ROJA-CAPUCHINOS,dish_es_v1_da687fa2af0b4205,churros,churros,8.0,5.0,1.0,7.0,1.0,0.0,0.8750,0.1250,0.0000,1.3752,1.5990,0.6325,0.6250,4.7500
1782,d04648f5-3214-49b1-ad99-f3c94aba4d82,Sloppy Joe´s Reina Mercedes,Bellavista - La Palmera,SECTOR SUR-LA PALMERA-REINA MERCEDES,dish_es_v1_1f6ccd2be75f1cc9,pizza,pizza,7.0,4.0,1.0,6.0,1.0,0.0,0.8571,0.1429,0.0000,0.9274,1.3746,0.4329,0.1429,3.8571
138,0fc81764-ec0b-4443-80f3-84e9971a8a96,Bar Blanco & Negro,Este - Alcosa - Torreblanca,TORREBLANCA,dish_es_v1_1f6ccd2be75f1cc9,pizza,pizza,7.0,5.0,1.0,5.0,1.0,1.0,0.7143,0.1429,0.1429,0.5929,0.7730,0.5093,0.4286,3.4286
125,0d1fdb30-33e3-4a8a-adf1-7661543fd6a7,Doña Emilia Manolo Mayo,Casco Antiguo,ARENAL,dish_es_v1_10850a8a8e314902,ensaladilla,ensaladilla,7.0,2.0,1.0,4.0,3.0,0.0,0.5714,0.4286,0.0000,0.8367,1.2950,0.5400,0.4286,3.2857
506,3b20f382-a6d0-4c7d-8e70-6d0ce11eee18,UDON Viapol,Nervión,LA BUHAIRA,dish_es_v1_830869f3d7f1fb36,ramen,ramen,7.0,1.0,1.0,1.0,1.0,5.0,0.1429,0.1429,0.7143,-1.2871,-1.6606,0.5793,0.5714,1.0000
2019,ea7d6564-2b10-4bef-97e9-f91cfb33d342,Pizzería San Pablo,San Pablo - Santa Justa,SAN PABLO D Y E,dish_es_v1_1f6ccd2be75f1cc9,pizza,pizza,6.0,5.0,1.0,6.0,0.0,0.0,1.0000,0.0000,0.0000,4.6146,5.1476,0.8350,0.8333,5.0000


In [19]:
# ============================================================
# 15. Casos representativos por etiqueta
# ============================================================

example_cols = [
    "place_name", "district_name", "neighborhood_name", "rating_value",
    "display_dish_name_es_v1", "dish_mention_text",
    "mention_sentiment_label_v1", "mention_sentiment_score_v1", "mention_sentiment_confidence_v1",
    "sentiment_reason_v1", "sentiment_flags_v1", "positive_terms_v1", "negative_terms_v1",
    "context_used_v1"
]

positive_examples = (
    mentions_sentiment_df[mentions_sentiment_df["mention_sentiment_label_v1"] == "positive"]
    .sort_values(["mention_sentiment_confidence_v1", "mention_sentiment_local_strength_v1"], ascending=[False, False])
    [example_cols]
    .head(40)
)

negative_examples = (
    mentions_sentiment_df[mentions_sentiment_df["mention_sentiment_label_v1"] == "negative"]
    .sort_values(["mention_sentiment_confidence_v1", "mention_sentiment_local_strength_v1"], ascending=[False, False])
    [example_cols]
    .head(40)
)

neutral_examples = (
    mentions_sentiment_df[mentions_sentiment_df["mention_sentiment_label_v1"] == "neutral"]
    .sort_values(["mention_sentiment_confidence_v1", "mention_sentiment_local_strength_v1"], ascending=[False, False])
    [example_cols]
    .head(40)
)

print("Ejemplos positivos")
display(positive_examples.head(10))

print("Ejemplos negativos")
display(negative_examples.head(10))

print("Ejemplos neutrales")
display(neutral_examples.head(10))


Ejemplos positivos


,place_name,district_name,neighborhood_name,rating_value,display_dish_name_es_v1,dish_mention_text,mention_sentiment_label_v1,mention_sentiment_score_v1,mention_sentiment_confidence_v1,sentiment_reason_v1,sentiment_flags_v1,positive_terms_v1,negative_terms_v1,context_used_v1
2385,Bar DE ARACENA A LA MESA,Este - Alcosa - Torreblanca,TORREBLANCA,5,carrillada,carrillada,positive,8.5377,0.95,local_context_primary,[intensifier_nearby],"[exquisita:positive_lexicon, riquisima:positive_lexicon, riquisimo:positive_lexicon, tierna:positive_lexicon]",[],"Nos recomendaron este lugar , y hemos acertado , todo estaba riquísimo , hemos pedido una carrillada súper tierna , con una salsa riquísima , un sáname elaborado súper interesante , una ensaladilla exquisita , y tienen"
1961,Adagio,Casco Antiguo,SAN GIL,4,albóndigas,albóndigas,positive,8.1740,0.95,local_context_primary,[intensifier_nearby],"[casero:positive_lexicon, en su punto:positive_lexicon, maravilloso:positive_lexicon, perfecto:positive_lexicon, rico:positive_lexicon]",[],"Pedimos dos tapas, las bravas espectaculares y el humus bastante rico y dos platos para compartir, los espaguetis norma y las albóndigas, los dos con un tomate casero maravilloso y en su punto perfecto todo."
1546,CASA MANOLO TAPAS BAR SEVILLANO,Casco Antiguo,ALFALFA,5,empanada,empanada,positive,7.9957,0.95,local_context_primary,[intensifier_nearby],"[buena:positive_lexicon, buenisimo:positive_lexicon, increible:positive_lexicon]",[],"on pisto y huevo son una delicia, el brioche de carrilleras estaba buenísimo (algo carillo), el solomillo al whisky una delicia, la empanada muy diferente y buena, increíble el salomojero con mojama y aguacate, la ensaladilla con gambas increíble, el pollo..."
2386,Bar DE ARACENA A LA MESA,Este - Alcosa - Torreblanca,TORREBLANCA,5,ensaladilla,ensaladilla,positive,7.8940,0.95,local_context_primary,[intensifier_nearby],"[exquisita:positive_lexicon, riquisima:positive_lexicon, riquisimo:positive_lexicon, tierna:positive_lexicon]",[],"o estaba riquísimo , hemos pedido una carrillada súper tierna , con una salsa riquísima , un sáname elaborado súper interesante , una ensaladilla exquisita , y tienen mucha variedad de carnes ibéricas , platos y postres que merecen ser degustados …y el ser..."
1378,Atlántico Sur,Norte,LAS NACIONES-PARQUE ATLANTICO-LAS DALIAS,4,pescado frito,pescado frito,positive,7.7537,0.95,local_context_primary,[intensifier_nearby],"[buena calidad:positive_lexicon, buena:positive_lexicon, calidad:positive_lexicon, fresco:positive_lexicon, riquisima:positive_lexicon]",[],"Disfrutamos muchísimo la experiencia en Atlántico Sur: la comida está riquísima y de muy buena calidad, especialmente el pescado frito, que se nota fresco y bien elaborado."
1572,"Venta Campo Rico ""El Arriero""",Norte,VALDEZORRAS,5,presa ibérica,presa ibérica,positive,7.6249,0.95,local_context_primary,[intensifier_nearby],"[buena:positive_lexicon, buenisima:positive_lexicon, espectacular:positive_lexicon, nos encanto:positive_lexicon]",[],"La comida estuvo espectacular: la carne, especialmente la presa ibérica de bellota, estaba muy buena, el revuelto de la casa nos encantó y la tarta de queso estaba buenísima."
1573,"Venta Campo Rico ""El Arriero""",Norte,VALDEZORRAS,5,tarta de queso,tarta de queso,positive,7.5860,0.95,local_context_primary,[intensifier_nearby],"[buena:positive_lexicon, buenisima:positive_lexicon, espectacular:positive_lexicon, nos encanto:positive_lexicon]",[],"La comida estuvo espectacular: la carne, especialmente la presa ibérica de bellota, estaba muy buena, el revuelto de la casa nos encantó y la tarta de queso estaba buenísima."
1545,CASA MANOLO TAPAS BAR SEVILLANO,Casco Antiguo,ALFALFA,5,solomillo al whisky,solomillo al whisky,positive,7.4940,0.95,local_context_primary,[intensifier_nearby],"[buena:positive_lexicon, buenisimo:positive_lexicon, increible:positive_lexicon, nos encanto:positive_lexicon]",[],", nos encantó todo, las alcachofas con pisto y huevo son una delicia, el br

Ejemplos negativos


,place_name,district_name,neighborhood_name,rating_value,display_dish_name_es_v1,dish_mention_text,mention_sentiment_label_v1,mention_sentiment_score_v1,mention_sentiment_confidence_v1,sentiment_reason_v1,sentiment_flags_v1,positive_terms_v1,negative_terms_v1,context_used_v1
1826,Bar Peña Sevillista San Jerónimo,Norte,SAN JERONIMO,1,calamares,calamares,negative,-6.5195,0.95,local_context_primary,[negation_detected],[sabor:negated_positive],"[congelado:negative_lexicon, fria:negative_lexicon, sabor:negated_positive, sin sabor:negative_lexicon]","La tortilla fría, y las papas congeladas de bastón, los calamares de paquete congelado sin sabor a saber si eran calamares."
1827,Bar Peña Sevillista San Jerónimo,Norte,SAN JERONIMO,1,calamares,calamares,negative,-6.5195,0.95,local_context_primary,[negation_detected],[sabor:negated_positive],"[congelado:negative_lexicon, fria:negative_lexicon, sabor:negated_positive, sin sabor:negative_lexicon]","La tortilla fría, y las papas congeladas de bastón, los calamares de paquete congelado sin sabor a saber si eran calamares."
1825,Bar Peña Sevillista San Jerónimo,Norte,SAN JERONIMO,1,tortilla de patatas,tortilla,negative,-5.8310,0.95,local_context_primary,[negation_detected],[sabor:negated_positive],"[congelado:negative_lexicon, fria:negative_lexicon, sabor:negated_positive, sin sabor:negative_lexicon]","La tortilla fría, y las papas congeladas de bastón, los calamares de paquete congelado sin sabor a saber si eran calamares."
1662,Cafe Bar las Leñas,Cerro - Amate,SANTA AURELIA-CANTABRICO-ATLANTICO-LA ROMERIA,1,bacalao,bacalao,negative,-4.7200,0.95,local_context_primary,[],[],"[crudo:negative_lexicon, malisimo:negative_lexicon]","Hemos pedido: Pimientos con melva: Los pimientos de bote y la melva prácticamente inexistente Patatas cheddar bacon: Patatas congeladas con queso malísimo y bacon crudo Pavías de bacalao: Mal hechas, crudas y congeladas."
1292,Taberna maestrante,Este - Alcosa - Torreblanca,PARQUE ALCOSA-JARDINES DEL EDÉN,1,croqueta,croquetas,negative,-4.3955,0.95,local_context_primary,[negation_detected],[sabor:negated_positive],"[grasienta:negative_lexicon, sabor:negated_positive, sin sabor:negative_lexicon]","La ensaladilla no era más que patata machacada con palitos de cangrejo y las croquetas un mazacote de masa grasienta sin sabor ninguno, refritas en aceite de motor."
1824,Bar Peña Sevillista San Jerónimo,Norte,SAN JERONIMO,1,carrillada,carrillada,negative,-4.3955,0.95,local_context_primary,[negation_detected],[sabor:negated_positive],"[dura:negative_lexicon, sabor:negated_positive, sin sabor:negative_lexicon]","La carrillada estaba dura, y sin sabor."
319,UDON Viapol,Nervión,LA BUHAIRA,1,ramen,Ramen,negative,-4.0495,0.95,local_context_primary,[negation_detected],[sabor:negated_positive],"[sabor:negated_positive, sin sabor:negative_lexicon]","Me pedí el Miso Ramen y los fideos parecían que estaban hervidos en agua, sin sabor alguno, rancios, y el caldo más de lo mismo, parecía de brick y sin sabor."
982,Ni Santa ni Clara,San Pablo - Santa Justa,SANTA CLARA,1,flamenquín,flamenquin,negative,-3.8940,0.95,local_context_primary,[],[],"[congelado:negative_lexicon, quemado:negative_lexicon]",El flamenquin venía quemado por fuera y congelado por dentro; primer plato devuelto.
2755,Bar Terracota,Macarena,DOCTOR BARRAQUER-GRUPO RENFE-POLICLINICO,1,presa ibérica,presa,negative,-3.8645,0.95,local_context_primary,[negation_detected],[merece la pena:negated_positive],"[merece la pena:negated_positive, no merece la pena:negative_lexicon]","Respecto a la comida, las bravas estaban buenas y la presa también, lo demás no merece la pena."
2822,Arcos 33,Los Remedios,LOS REMEDIOS,1,carrillada,carrillada,negative,-3.8586,0.95,local_context_primary,[intensifier_nearby],[],"[grasienta:negative_lexicon, pasada:negative_lexicon]","Pedimos variedad y entre ellas carrillada, la note pasada y muy grasienta y le comenté que a todos los lugares que veo que la tienen en carta la pido es un plato que me gusta m

Ejemplos neutrales


,place_name,district_name,neighborhood_name,rating_value,display_dish_name_es_v1,dish_mention_text,mention_sentiment_label_v1,mention_sentiment_score_v1,mention_sentiment_confidence_v1,sentiment_reason_v1,sentiment_flags_v1,positive_terms_v1,negative_terms_v1,context_used_v1
1938,Bar Restaurante La Cochera,Cerro - Amate,EL CERRO,3,tarta de queso,tarta de queso,neutral,-0.0622,0.78,local_context_primary,"[mixed_local_evidence, negation_detected]","[buenisimo:positive_lexicon, calidad:positive_lexicon, sabrosa:negated_positive]","[congelada:negative_lexicon, sabrosa:negated_positive]","La verdad es que no podemos tener queja alguna de la calidad de la comida, estaba todo buenísimo, menos los postres: la tarta de queso caliente y poco sabrosa y la tarta de Lotus congelada literalmente."
1939,Bar Restaurante La Cochera,Cerro - Amate,EL CERRO,3,tarta de queso,tarta,neutral,-0.0622,0.78,local_context_primary,"[mixed_local_evidence, negation_detected]","[buenisimo:positive_lexicon, calidad:positive_lexicon, sabrosa:negated_positive]","[congelada:negative_lexicon, sabrosa:negated_positive]","La verdad es que no podemos tener queja alguna de la calidad de la comida, estaba todo buenísimo, menos los postres: la tarta de queso caliente y poco sabrosa y la tarta de Lotus congelada literalmente."
1776,Indiano Plaza,Casco Antiguo,ENCARNACIÓN-REGINA,5,croqueta,croquetas,neutral,0.2360,0.78,local_context_primary,[mixed_local_evidence],[buenisima:positive_lexicon],[seco:negative_lexicon],"El sitio precioso y la comida buenísima, pedí las croquetas de sardinas con tomate seco y el brioche napolitano."
603,Tarannà,Casco Antiguo,SANTA CATALINA,5,brownie,brownie,neutral,-0.2006,0.78,local_context_primary,"[mixed_local_evidence, negation_detected]","[espectacular:negated_positive, perfecta:positive_lexicon]",[espectacular:negated_positive],"Y para acabar la tarta de queso templada que era perfecta y un brownie que nos dejó sin palabras, espectacular."
1249,Bar Antojo,Casco Antiguo,SAN GIL,4,patatas bravas,bravas,neutral,-0.5391,0.78,local_context_primary,"[intensifier_nearby, mixed_local_evidence, negation_detected]","[bueno:positive_lexicon, jugoso:negated_positive, sabroso:negated_positive]","[jugoso:negated_positive, sabroso:negated_positive]","Estaba todo muy bueno, destacando la cáscara y las bravas, por otro lado, el solomillo al Whisky no lo recomendaria, no estaba jugoso y sabroso."
602,Tarannà,Casco Antiguo,SANTA CATALINA,5,tarta de queso,tarta de queso,neutral,0.1360,0.78,local_context_primary,"[mixed_local_evidence, negation_detected]","[espectacular:negated_positive, perfecta:positive_lexicon]",[espectacular:negated_positive],"Y para acabar la tarta de queso templada que era perfecta y un brownie que nos dejó sin palabras, espectacular."
357,Barrabas,Casco Antiguo,MUSEO,4,torrija,torrija,neutral,-0.3351,0.78,local_context_primary,"[intensifier_nearby, mixed_local_evidence]",[bueno:positive_lexicon],[seco:negative_lexicon],El pollo un poquito seco y el postre (torrija) GOD mal muy muy bueno.
1615,Restaurante La Herrería,Bellavista - La Palmera,ELCANO-BERMEJALES,4,solomillo al whisky,solomillo,neutral,0.1300,0.78,local_context_primary,[mixed_local_evidence],[buenisimo:positive_lexicon],[pasado:negative_lexicon],Incluso el solomillo la mitad lo hemos pasado más y también buenísimo!
353,Gascona,Triana,TRIANA CASCO ANTIGUO,4,ensaladilla,ensaladilla,neutral,0.2551,0.78,local_context_primary,"[intensifier_nearby, mixed_local_evidence]",[buenisimo:positive_lexicon],[frio:negative_lexicon],Esta vez degustamos: Una media de ensaladilla de pulpo Una media de calamares Un arroz caldoso de mar y montaña que está buenísimo y apetecía porque el día estaba muy frío.
354,Gascona,Triana,TRIANA CASCO ANTIGUO,4,pulpo,pulpo,neutral,0.2551,0.78,local_context_primary,"[intensifier_nearby, mixed_local_evidence]",[buenisimo:positive_lexicon],[frio:negative_lexicon],Esta vez degustamos: Una media de ensaladilla de pulpo Una media de calamares Un arroz caldoso de

In [20]:
# ============================================================
# 16. Exportar artefactos
# ============================================================

OUTPUT_JSONL = OUTPUT_DIR / "sevilla_dish_mentions_with_sentiment_v1.jsonl"
OUTPUT_CSV = OUTPUT_DIR / "sevilla_dish_mentions_with_sentiment_v1.csv"
MANUAL_REVIEW_CSV = OUTPUT_DIR / "sevilla_dish_mentions_sentiment_manual_review_v1.csv"
SUMMARY_JSON = OUTPUT_DIR / "sevilla_mention_sentiment_summary_v1.json"

BY_DISH_CSV = OUTPUT_DIR / "sevilla_mention_sentiment_by_dish_v1.csv"
BY_DISTRICT_CSV = OUTPUT_DIR / "sevilla_mention_sentiment_by_district_v1.csv"
BY_NEIGHBORHOOD_CSV = OUTPUT_DIR / "sevilla_mention_sentiment_by_neighborhood_v1.csv"
BY_PLACE_DISH_CSV = OUTPUT_DIR / "sevilla_mention_sentiment_by_place_dish_v1.csv"

POSITIVE_EXAMPLES_CSV = OUTPUT_DIR / "sevilla_mention_sentiment_positive_examples_v1.csv"
NEGATIVE_EXAMPLES_CSV = OUTPUT_DIR / "sevilla_mention_sentiment_negative_examples_v1.csv"
NEUTRAL_EXAMPLES_CSV = OUTPUT_DIR / "sevilla_mention_sentiment_neutral_examples_v1.csv"
FLAGS_CSV = OUTPUT_DIR / "sevilla_mention_sentiment_flag_counts_v1.csv"
TERMS_CSV = OUTPUT_DIR / "sevilla_mention_sentiment_term_counts_v1.csv"

# CSV principal
mentions_sentiment_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

# JSONL principal
write_jsonl(dataframe_to_records(mentions_sentiment_df), OUTPUT_JSONL)

# Manual review
manual_review_df[manual_cols].to_csv(MANUAL_REVIEW_CSV, index=False, encoding="utf-8")

# Resúmenes
summary_by_dish.to_csv(BY_DISH_CSV, index=False, encoding="utf-8")
summary_by_district.to_csv(BY_DISTRICT_CSV, index=False, encoding="utf-8")
summary_by_neighborhood.to_csv(BY_NEIGHBORHOOD_CSV, index=False, encoding="utf-8")
summary_by_place_dish.to_csv(BY_PLACE_DISH_CSV, index=False, encoding="utf-8")

positive_examples.to_csv(POSITIVE_EXAMPLES_CSV, index=False, encoding="utf-8")
negative_examples.to_csv(NEGATIVE_EXAMPLES_CSV, index=False, encoding="utf-8")
neutral_examples.to_csv(NEUTRAL_EXAMPLES_CSV, index=False, encoding="utf-8")

flag_counts.rename_axis("flag").reset_index(name="count").to_csv(FLAGS_CSV, index=False, encoding="utf-8")

term_counts_df = pd.concat([
    positive_term_counts.rename("positive_count"),
    negative_term_counts.rename("negative_count"),
], axis=1).fillna(0).reset_index().rename(columns={"index": "term"})
term_counts_df.to_csv(TERMS_CSV, index=False, encoding="utf-8")

print("Artefactos exportados en:", OUTPUT_DIR)
print("JSONL principal:", OUTPUT_JSONL)
print("CSV principal:", OUTPUT_CSV)


Artefactos exportados en: c:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline\data\artifacts\ai\sevilla\sentiment
JSONL principal: c:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline\data\artifacts\ai\sevilla\sentiment\sevilla_dish_mentions_with_sentiment_v1.jsonl
CSV principal: c:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline\data\artifacts\ai\sevilla\sentiment\sevilla_dish_mentions_with_sentiment_v1.csv


In [21]:
# ============================================================
# 17. Summary final y checks
# ============================================================

label_counts = mentions_sentiment_df["mention_sentiment_label_v1"].value_counts().to_dict()
reason_counts = mentions_sentiment_df["sentiment_reason_v1"].value_counts().to_dict()

checks = {
    "has_mentions": len(mentions_sentiment_df) > 0,
    "all_mentions_have_sentiment_label": mentions_sentiment_df["mention_sentiment_label_v1"].notna().all(),
    "all_mentions_have_sentiment_confidence": mentions_sentiment_df["mention_sentiment_confidence_v1"].notna().all(),
    "all_mentions_have_sentiment_reason": mentions_sentiment_df["sentiment_reason_v1"].notna().all(),
    "mention_ids_are_unique": mentions_sentiment_df["mention_id"].nunique() == len(mentions_sentiment_df),
    "has_positive_mentions": (mentions_sentiment_df["mention_sentiment_label_v1"] == "positive").any(),
    "has_neutral_mentions": (mentions_sentiment_df["mention_sentiment_label_v1"] == "neutral").any(),
    "has_negative_mentions": (mentions_sentiment_df["mention_sentiment_label_v1"] == "negative").any(),
    "has_local_context_primary": (mentions_sentiment_df["sentiment_reason_v1"] == "local_context_primary").any(),
    "has_review_prior_fallback": (mentions_sentiment_df["sentiment_reason_v1"] == "review_prior_fallback").any(),
    "output_jsonl_exists": OUTPUT_JSONL.exists(),
    "output_csv_exists": OUTPUT_CSV.exists(),
}

summary = {
    "notebook": NOTEBOOK_NAME,
    "version": VERSION,
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "input_path": str(INPUT_PATH),
    "output_dir": str(OUTPUT_DIR),
    "input": {
        "total_mentions": int(len(df)),
        "unique_reviews": int(df["review_id"].nunique()),
        "unique_places": int(df["place_id"].nunique()),
        "unique_dishes": int(df["dish_id_v1"].nunique()),
        "unique_neighborhoods": int(df["neighborhood_id"].nunique()) if "neighborhood_id" in df.columns else None,
        "unique_districts": int(df["district_id"].nunique()) if "district_id" in df.columns else None,
    },
    "sentiment": {
        "total_mentions_with_sentiment": int(len(mentions_sentiment_df)),
        "mention_sentiment_counts": {str(k): int(v) for k, v in label_counts.items()},
        "sentiment_reason_counts": {str(k): int(v) for k, v in reason_counts.items()},
        "manual_review_mentions": int(len(manual_review_df)),
        "confidence": {
            "min": float(mentions_sentiment_df["mention_sentiment_confidence_v1"].min()),
            "mean": float(mentions_sentiment_df["mention_sentiment_confidence_v1"].mean()),
            "median": float(mentions_sentiment_df["mention_sentiment_confidence_v1"].median()),
            "max": float(mentions_sentiment_df["mention_sentiment_confidence_v1"].max()),
        },
        "score": {
            "min": float(mentions_sentiment_df["mention_sentiment_score_v1"].min()),
            "mean": float(mentions_sentiment_df["mention_sentiment_score_v1"].mean()),
            "median": float(mentions_sentiment_df["mention_sentiment_score_v1"].median()),
            "max": float(mentions_sentiment_df["mention_sentiment_score_v1"].max()),
        },
        "flag_counts": {str(k): int(v) for k, v in flag_counts.to_dict().items()},
        "top_positive_terms": {str(k): int(v) for k, v in positive_term_counts.head(30).to_dict().items()},
        "top_negative_terms": {str(k): int(v) for k, v in negative_term_counts.head(30).to_dict().items()},
    },
    "summaries": {
        "dish_rows": int(len(summary_by_dish)),
        "district_rows": int(len(summary_by_district)),
        "neighborhood_rows": int(len(summary_by_neighborhood)),
        "place_dish_rows": int(len(summary_by_place_dish)),
        "top_dishes_by_mentions": dataframe_to_records(summary_by_dish.head(30)),
    },
    "checks": checks,
    "artifacts": {
        "mentions_jsonl": OUTPUT_JSONL.name,
        "mentions_csv": OUTPUT_CSV.name,
        "manual_review_csv": MANUAL_REVIEW_CSV.name,
        "summary_by_dish_csv": BY_DISH_CSV.name,
        "summary_by_district_csv": BY_DISTRICT_CSV.name,
        "summary_by_neighborhood_csv": BY_NEIGHBORHOOD_CSV.name,
        "summary_by_place_dish_csv": BY_PLACE_DISH_CSV.name,
        "positive_examples_csv": POSITIVE_EXAMPLES_CSV.name,
        "negative_examples_csv": NEGATIVE_EXAMPLES_CSV.name,
        "neutral_examples_csv": NEUTRAL_EXAMPLES_CSV.name,
        "flags_csv": FLAGS_CSV.name,
        "terms_csv": TERMS_CSV.name,
    },
}

save_json(summary, SUMMARY_JSON)

print(json.dumps(summary, ensure_ascii=False, indent=2, default=str)[:5000])
print("\nSummary guardado en:", SUMMARY_JSON)


{
  "notebook": "15_sevilla_mention_sentiment",
  "version": "sevilla_mention_sentiment_hybrid_v1",
  "generated_at": "2026-05-10T22:06:32.917508+00:00",
  "input_path": "c:\\Users\\USUARIO\\Documents\\Proyectos_Master_IA_Big_Data\\hidden-gems-pipeline\\data\\artifacts\\ai\\sevilla\\dish_normalization\\sevilla_dish_candidates_ranking_ready_v1.jsonl",
  "output_dir": "c:\\Users\\USUARIO\\Documents\\Proyectos_Master_IA_Big_Data\\hidden-gems-pipeline\\data\\artifacts\\ai\\sevilla\\sentiment",
  "input": {
    "total_mentions": 2979,
    "unique_reviews": 1452,
    "unique_places": 635,
    "unique_dishes": 181,
    "unique_neighborhoods": 94,
    "unique_districts": 11
  },
  "sentiment": {
    "total_mentions_with_sentiment": 2979,
    "mention_sentiment_counts": {
      "positive": 2395,
      "negative": 388,
      "neutral": 196
    },
    "sentiment_reason_counts": {
      "local_context_primary": 1590,
      "review_prior_fallback": 1389
    },
    "manual_review_mentions": 1896,
  

## 18. Interpretación y siguiente paso

Este notebook produce una señal de sentimiento por mención, no un ranking final.

El siguiente paso del flujo será:

```text
16_sevilla_place_dish_signals.ipynb
```

Ese notebook agregará las menciones por `place_id + dish_id_v1` y calculará señales como:

- número de menciones;
- número de reviews;
- ratio positivo/neutral/negativo;
- score bayesiano o suavizado;
- evidencia mínima para ranking;
- calidad de señal;
- preparación para `sevilla_pilot`.